## Phase 3, Attempt 1: class_weight='balanced'

The baseline treated every row equally during training, even though
legit outnumbers fraud ~578:1. `class_weight='balanced'` reweights
the loss so each class contributes equally in aggregate, fraud rows
get upweighted ~578x, legit rows get downweighted slightly.

Everything else (split, features, threshold) stays identical to the
baseline, so any change in the metrics is attributable to this one
lever. Expectation: recall up, precision down, the classic tradeoff,
made visible.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
)

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())  # sanity check: should both be ~0.0017, matching baseline

(227845, 30) (56962, 30)
0.001729245759178389 0.0017204452090867595


In [ ]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)
y_scores_balanced = model_balanced.predict_proba(X_test)[:, 1]

cm_balanced = confusion_matrix(y_test, y_pred_balanced)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_balanced)

precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
auprc_balanced = average_precision_score(y_test, y_scores_balanced)

print(f"\nPrecision: {precision_balanced:.4f}  (baseline was 0.8312)")
print(f"Recall:    {recall_balanced:.4f}  (baseline was 0.6531)")
print(f"AUPRC:     {auprc_balanced:.4f}  (baseline was 0.7427)")

Confusion matrix [[TN, FP], [FN, TP]]:
[[55437  1427]
 [    8    90]]

Precision: 0.0593  (baseline was 0.8312)
Recall:    0.9184  (baseline was 0.6531)
AUPRC:     0.7082  (baseline was 0.7427)


/home/enio30/Desktop/fraudlens/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 5000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=5000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
